- chunk strategy
    - 按照固定的 chunk_token_size（默认 1200）进行切分。
    - 切分时保留 chunk_overlap_token_size（默认 100）的重叠内容，以保证上下文连贯性。
    - 公式: start 从 0 开始，每次步进 chunk_token_size - chunk_overlap_token_size。

### KG-based RAG 

> from chunk-based rag （naive） to kg-based rag （hybrid or mix）

- 必要性场景（海量的文档），某种意义上实现全局性
    - 多跳推理
    - 计数
    - 。。。

| 维度 | Chunk-based (Naive) | KG-based (LightRAG) | 核心价值 |
| :--- | :--- | :--- | :--- |
| **数据形态** | 碎片化的“点” (Vector Chunks) | 连通的“网” (Graph: Entities & Relations) | **完整性**：将离散信息重组为结构化知识 |
| **推理方式** | 隐式推理：靠 LLM 在有限 Context 窗口内基于碎片硬推 | 显式推理：利用图结构进行多跳路径游走和关联发现 | **深度与可解释性**：逻辑链条清晰，可跨文档推理 |
| **统计与全局** | 极弱：受限于 Top-K 截断，存在严重的幸存者偏差 | 较强：通过 Global Query 提取宏观主题，支持结构化聚合 | **宏观洞察**：能回答“总结”、“趋势”、“计数”类问题 |
| **抗噪与消歧** | 弱：易被字面相似但语义无关的文档干扰 | 强：实体唯一ID和图谱聚类特性天然过滤无关噪音 | **精准度**：同名异义词消歧，冲突信息检测 |

- 分析：log 输出，以及 langfuse api input / output

### Merge

> https://github.com/HKUDS/LightRAG/blob/main/lightrag/operate.py#L2398

> Upsert 是 Update（更新）和 Insert（插入）

- 实体对齐（Entity Alignment）和同义词匹配主要依赖于 LLM 的生成一致性，而不是后期的向量化模糊匹配或专门的对齐模型。这意味着："Apple Inc." 和 "Apple" 在 LightRAG 眼里是两个完全不同的节点，除非 LLM 在提取时明确将它们归一化。
    - entity_extraction_system_prompt
        - `entity_name: The name of the entity. If the entity name is case-insensitive, capitalize the first letter of each significant word (title case). Ensure **consistent naming** across the entire extraction process.`
- 实体合并
    - 个实体或关系在不同上下文中会有不同的侧面，因此它的逻辑是将这些侧面叠加（通过 source_id 列表、描述列表、权重累加），并在必要时（如描述过长）使用 LLM 进行语义压缩。这保证了知识图谱既紧凑又包含了丰富的上下文信息。
- 关系合并 (Relationships)
    - Edge Description 的合并策略: 它不仅仅是简单的字符串拼接，而是采用了一个智能摘要 (Map-Reduce) 策略：
        - 收集 (Collect)：将所有从不同 Chunk 中提取到的关于这对关系的描述收集起来，再加上数据库里已有的描述 (description_list = already_description + sorted_descriptions)。
        - 去重 (Deduplicate)：在收集前通过 unique_edges 字典对完全相同的描述文本进行了去重。
        - 判断 (Decision):
            - 如果短：直接用分隔符 `<SEP>` 拼接。
            - 如果长（超过 token 限制）：调用 LLM 进行语义摘要 (Summarization)，生成一段精炼的描述。
                - https://github.com/HKUDS/LightRAG/blob/main/lightrag/prompt.py
                - `summarize_entity_descriptions`

In [2]:
# Alice (Engineer), Bob (Manager), Alice->Bob (Colleague)
chunk1_results = (
    {
        "Alice": [{"entity_name": "Alice", "entity_type": "Person", "description": "An engineer", "source_id": "chunk_1"}],
        "Bob": [{"entity_name": "Bob", "entity_type": "Person", "description": "A manager", "source_id": "chunk_1"}]
    },
    {
        ("Alice", "Bob"): [{"weight": 1, "keywords": "colleague,work", "description": "Alice works with Bob"}]
    }
)
# 模拟重复和增量信息 
# 包含: Alice (Lives in NY), Alice->Bob (Friends)
chunk2_results = (
    {
        "Alice": [{"entity_name": "Alice", "entity_type": "Person", "description": "Lives in NY", "source_id": "chunk_2"}]
    },
    {
        ("Alice", "Bob"): [{"weight": 2, "keywords": "friend", "description": "Alice is friends with Bob"}]
    }
)

In [1]:
!python scripts/merge.py


--- [Run 1] Processing Chunk 1 ---
=== STARTING MERGE PROCESS ===

>>> PHASE 1: MERGING NODES

--- Processing Entity: Alice ---
  [DB] Upsert Node: Alice | Data: {'entity_name': 'Alice', 'entity_type': 'Person', 'description': 'An engineer', 'source_id': 'chunk_1'}

--- Processing Entity: Bob ---
  [DB] Upsert Node: Bob | Data: {'entity_name': 'Bob', 'entity_type': 'Person', 'description': 'A manager', 'source_id': 'chunk_1'}

>>> PHASE 2: MERGING EDGES

--- Processing Edge: Alice <-> Bob ---
  [DB] Upsert Edge: ('Alice', 'Bob') | Data: {'source': 'Alice', 'target': 'Bob', 'weight': 1, 'keywords': 'colleague,work', 'description': 'Alice works with Bob'}

--- [Run 2] Processing Chunk 2 (Incremental) ---
=== STARTING MERGE PROCESS ===

>>> PHASE 1: MERGING NODES

--- Processing Entity: Alice ---
  Found existing node: {'entity_name': 'Alice', 'entity_type': 'Person', 'description': 'An engineer', 'source_id': 'chunk_1'}
  [DB] Upsert Node: Alice | Data: {'entity_name': 'Alice', 'entity_

### KG 可视化

- https://github.com/HKUDS/LightRAG/blob/main/examples/graph_visual_with_html.py

### Retrieval

> lightrag 支持多种 retrieval mode (Query Param)

- naive vs. hybrid vs. mix
    - naive: 仅向量，vector-retrieval
        - query & chunks_vdb
    - hybrid: kg (local + global)
        - 纯粹基于 知识图谱 (KG)。它结合了 Local (特定实体邻居) 和 Global (全局关系) 的图谱信息，但不直接去向量库里找原始文本块 (Chunks)。
    - mix: kg & vector-retrieval
        - 是 Hybrid 加上 Naive。它既利用了 KG 的推理能力，又额外执行了一次传统的向量检索来获取原始文本块。这是“最重”但也最全面的模式。

#### local & global

- local query: 基于实体 (Entity) 的邻域检索。关注细节和具体实体的直接关联。
    - Low-level Keywords: LLM 从 Query 中提取出具体的实体名称（例如 "Alice", "iPhone 15"）。
    - Vector Search (Entities): 用这些关键词在 entities_vdb（实体向量库）中检索相似的实体节点。
    - Graph Traversal (1-hop): 找到这些实体后，去图数据库 (knowledge_graph_inst) 中获取它们的所有直接邻居关系（即 _find_most_related_edges_from_entities）。
    - 结果：返回这些实体 + 它们的一度关系。
- Global Query: 基于关系 (Relationship) 的全局检索。关注宏观主题、跨实体的连接模式。
    - High-level Keywords: LLM 从 Query 中提取出更抽象的主题或关系描述（例如 "conflict", "collaboration"）。
    - Vector Search (Relationships): 用这些关键词在 relationships_vdb（关系向量库）中检索相似的边。
        - 注意：LightRAG 独特之处在于它对 Edge 也做了 Embedding。
    - Entity Retrieval: 找到这些边后，再去获取这些边两端的实体 (_find_most_related_entities_from_relationships)。
    - 结果：返回这些关系 + 相关的实体。